# Trace Anatomy

Audits the Trace public manifest against a live tiny-model capture.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_grads=True,
    save_arg_values=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].out.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.ops.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.ops.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer ops, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## Trace Anatomy: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.Trace",
    "tl.Trace.DEFAULT_FILL_STATE",
    "tl.Trace.FIELD_FORK_POLICY",
    "tl.Trace.PORTABLE_STATE_SPEC",
    "tl.Trace._out_hash_cache",
    "tl.Trace._out_recipe_revision",
    "tl.Trace._layers_logged",
    "tl.Trace._layers_saved",
    "tl.Trace._append_sequence_id",
    "tl.Trace._final_to_raw_layer_labels",
    "tl.Trace._grad_layer_nums_to_save",
    "tl.Trace._has_direct_writes",
    "tl.Trace._intervention_spec",
    "tl.Trace._last_hook_handle_ids",
    "tl.Trace._layer_counter",
    "tl.Trace._layer_num_to_lookup_keys_dict",
    "tl.Trace._layer_nums_to_save",
    "tl.Trace._layers_where_internal_branches_merge_with_input",
    "tl.Trace._lookup_keys_to_layer_num_dict",
    "tl.Trace._tracing_finished",
    "tl.Trace._raw_layer_dict",
    "tl.Trace._raw_layer_labels_list",
    "tl.Trace._raw_layer_type_counter",
    "tl.Trace._raw_to_final_layer_labels",
    "tl.Trace._source_code_blob",
    "tl.Trace._source_model_ref",
    "tl.Trace._spec_revision",
    "tl.Trace._unsaved_layers_lookup_keys",
    "tl.Trace._warned_direct_write",
    "tl.Trace._warned_mutate_in_place",
    "tl.Trace.out_postfunc",
    "tl.Trace._out_transform_repr",
    "tl.Trace.out_transform",
    "tl.Trace.add_node_overlay",
    "tl.Trace.animate_ops",
    "tl.Trace.append_run_state_from",
    "tl.Trace.attach_hooks",
    "tl.Trace.backward_memory_backend",
    "tl.Trace.backward_num_calls",
    "tl.Trace.backward_peak_memory",
    "tl.Trace.backward_root_grad_fn_id",
    "tl.Trace.buffer_layers",
    "tl.Trace.buffer_num_calls",
    "tl.Trace.buffers",
    "tl.Trace.capture_cache_hit",
    "tl.Trace.capture_cache_key",
    "tl.Trace.capture_cache_path",
    "tl.Trace.capture_args_template",
    "tl.Trace.trace_annotations",
    "tl.Trace.check_metadata_invariants",
    "tl.Trace.cleanup",
    "tl.Trace.clear_hooks",
    "tl.Trace.conditional_arm_entry_edges",
    "tl.Trace.conditional_branch_edges",
    "tl.Trace.conditional_edge_call_indices",
    "tl.Trace.conditional_elif_entry_edges",
    "tl.Trace.conditional_else_entry_edges",
    "tl.Trace.conditional_records",
    "tl.Trace.conditional_then_entry_edges",
    "tl.Trace._current_func_barcode",
    "tl.Trace.detach_hooks",
    "tl.Trace.detach_saved_activations",
    "tl.Trace.recurrence_detection",
    "tl.Trace.do",
    "tl.Trace.emit_nvtx",
    "tl.Trace.equivalent_ops",
    "tl.Trace.find_layers",
    "tl.Trace.find_sites",
    "tl.Trace.first_nonfinite",
    "tl.Trace.flops_by_type",
    "tl.Trace.fork",
    "tl.Trace.forward_source_line",
    "tl.Trace.grad_fn_logs",
    "tl.Trace.grad_fn_order",
    "tl.Trace.grad_fns",
    "tl.Trace.grad_transform",
    "tl.Trace.grad_transform_repr",
    "tl.Trace.grads_to_save",
    "tl.Trace.graph_shape_hash",
    "tl.Trace.has_backward_pass",
    "tl.Trace.has_conditional_branching",
    "tl.Trace.has_grads",
    "tl.Trace.input_id",
    "tl.Trace.input_layers",
    "tl.Trace.input_annotations",
    "tl.Trace.input_shape_hash",
    "tl.Trace.internal_source_ops",
    "tl.Trace.internally_terminated_bool_ops",
    "tl.Trace.internal_sink_ops",
    "tl.Trace.intervention_ready",
    "tl.Trace.intervention_spec",
    "tl.Trace.io_format_version",
    "tl.Trace.is_appended",
    "tl.Trace.is_branching",
    "tl.Trace.is_recurrent",
    "tl.Trace.keep_unsaved_layers",
    "tl.Trace.last_run_ctx",
    "tl.Trace.last_run_records",
    "tl.Trace.layer_dict_all_keys",
    "tl.Trace.layer_dict_main_keys",
    "tl.Trace.layer_labels",
    "tl.Trace.layer_labels",
    "tl.Trace.op_labels",
    "tl.Trace.layer_list",
    "tl.Trace.layer_logs",
    "tl.Trace.layer_num_calls",
    "tl.Trace.layers",
    "tl.Trace.layers_with_params",
    "tl.Trace.ops_with_saved_outs",
    "tl.Trace.ops_with_saved_grads",
    "tl.Trace.load",
    "tl.Trace.log_backward",
    "tl.Trace.capture_mode",
    "tl.Trace.macs_by_type",
    "tl.Trace.manual_tensor_connections",
    "tl.Trace.mark_layer_depths",
    "tl.Trace.max_recurrent_loops",
    "tl.Trace.model_name",
    "tl.Trace.module_filter",
    "tl.Trace.modules",
    "tl.Trace.name",
    "tl.Trace.num_context_lines",
    "tl.Trace.num_grad_fns",
    "tl.Trace.num_grad_fns_without_op",
    "tl.Trace.num_ops",
    "tl.Trace.num_streamed_ops",
    "tl.Trace.num_saved_ops",
    "tl.Trace.num_tensors",
    "tl.Trace.observer_spans",
    "tl.Trace.ledger",
    "tl.Trace.orphan_ops",
    "tl.Trace.output_device",
    "tl.Trace.output_layers",
    "tl.Trace.param_logs",
    "tl.Trace.params",
    "tl.Trace.parent_run",
    "tl.Trace.end_time",
    "tl.Trace.start_time",
    "tl.Trace.preview_fastlog",
    "tl.Trace.raise_on_nan",
    "tl.Trace.random_seed",
    "tl.Trace.recording_backward",
    "tl.Trace.recording_kept",
    "tl.Trace.relationship_evidence",
    "tl.Trace.release_param_refs",
    "tl.Trace.remove",
    "tl.Trace.render_dagua_graph",
    "tl.Trace.draw",
    "tl.Trace.replace_run_state_from",
    "tl.Trace.replay",
    "tl.Trace.replay_from",
    "tl.Trace.report_values",
    "tl.Trace.rerun",
    "tl.Trace.resolve_sites",
    "tl.Trace.root_module",
    "tl.Trace.run_state",
    "tl.Trace.save",
    "tl.Trace.save_arg_values",
    "tl.Trace.save_grads",
    "tl.Trace.save_intervention",
    "tl.Trace.save_new_outs",
    "tl.Trace.save_raw_outs",
    "tl.Trace.save_raw_grads",
    "tl.Trace.save_rng_states",
    "tl.Trace.save_code_context",
    "tl.Trace.saved_out_memory",
    "tl.Trace.saved_out_memory_str",
    "tl.Trace.set",
    "tl.Trace.show",
    "tl.Trace.draw_backward",
    "tl.Trace.model_class",
    "tl.Trace.model_id",
    "tl.Trace.streaming_pass_logs",
    "tl.Trace.suggest",
    "tl.Trace.summary",
    "tl.Trace.cleanup_duration",
    "tl.Trace.forward_duration",
    "tl.Trace.func_calls_duration",
    "tl.Trace.overhead_duration",
    "tl.Trace.setup_duration",
    "tl.Trace.duration",
    "tl.Trace.to_csv",
    "tl.Trace.to_dagua_graph",
    "tl.Trace.to_json",
    "tl.Trace.to_pandas",
    "tl.Trace.to_parquet",
    "tl.Trace.total_out_memory",
    "tl.Trace.total_out_memory_str",
    "tl.Trace.autograd_saved_memory",
    "tl.Trace.total_flops",
    "tl.Trace.total_flops_backward",
    "tl.Trace.total_flops_forward",
    "tl.Trace.total_macs",
    "tl.Trace.total_macs_backward",
    "tl.Trace.total_macs_forward",
    "tl.Trace.num_layers_with_params",
    "tl.Trace.num_param_tensors",
    "tl.Trace.num_params",
    "tl.Trace.num_params_frozen",
    "tl.Trace.param_memory",
    "tl.Trace.param_memory_str",
    "tl.Trace.num_params_trainable",
    "tl.Trace.train_mode",
    "tl.Trace.uncalled_modules",
    "tl.Trace.unlogged_ops",
    "tl.Trace.unsupported_ops",
    "tl.Trace.validate_forward_pass",
    "tl.Trace.validate_saved_outs",
    "tl.Trace.verbose",
    "tl.Trace.visualization_field_audit",
    "tl.Trace.param_hash_quick",
    "tl.Trace.param_hash_full",
]

pretty_print_fields(
    log,
    [
        "model_name",
        "model_class_str",
        "num_layers",
        "num_ops",
        "num_calls_run",
        "duration",
    ],
)
print("contains linear", "linear_1_1" in log)
print("iter first", next(iter(log)).layer_label)

## Identity and timing

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.Trace",
    "tl.Trace.DEFAULT_FILL_STATE",
    "tl.Trace.FIELD_FORK_POLICY",
    "tl.Trace.PORTABLE_STATE_SPEC",
    "tl.Trace._out_hash_cache",
    "tl.Trace._out_recipe_revision",
    "tl.Trace._layers_logged",
    "tl.Trace._layers_saved",
    "tl.Trace._append_sequence_id",
    "tl.Trace._final_to_raw_layer_labels",
    "tl.Trace._grad_layer_nums_to_save",
    "tl.Trace._has_direct_writes",
    "tl.Trace._intervention_spec",
    "tl.Trace._last_hook_handle_ids",
    "tl.Trace._layer_counter",
    "tl.Trace._layer_num_to_lookup_keys_dict",
    "tl.Trace._layer_nums_to_save",
    "tl.Trace._layers_where_internal_branches_merge_with_input",
    "tl.Trace._lookup_keys_to_layer_num_dict",
    "tl.Trace._tracing_finished",
    "tl.Trace._raw_layer_dict",
    "tl.Trace._raw_layer_labels_list",
    "tl.Trace._raw_layer_type_counter",
    "tl.Trace._raw_to_final_layer_labels",
    "tl.Trace._source_code_blob",
    "tl.Trace._source_model_ref",
    "tl.Trace._spec_revision",
    "tl.Trace._unsaved_layers_lookup_keys",
    "tl.Trace._warned_direct_write",
    "tl.Trace._warned_mutate_in_place",
    "tl.Trace.out_postfunc",
    "tl.Trace._out_transform_repr",
    "tl.Trace.out_transform",
    "tl.Trace.add_node_overlay",
    "tl.Trace.animate_ops",
    "tl.Trace.append_run_state_from",
]

audit_touch_items("Identity and timing", ITEMS, AUDIT_CONTEXT)

## Layer and module access

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.Trace.attach_hooks",
    "tl.Trace.backward_memory_backend",
    "tl.Trace.backward_num_calls",
    "tl.Trace.backward_peak_memory",
    "tl.Trace.backward_root_grad_fn_id",
    "tl.Trace.buffer_layers",
    "tl.Trace.buffer_num_calls",
    "tl.Trace.buffers",
    "tl.Trace.capture_cache_hit",
    "tl.Trace.capture_cache_key",
    "tl.Trace.capture_cache_path",
    "tl.Trace.capture_args_template",
    "tl.Trace.trace_annotations",
    "tl.Trace.check_metadata_invariants",
    "tl.Trace.cleanup",
    "tl.Trace.clear_hooks",
    "tl.Trace.conditional_arm_entry_edges",
    "tl.Trace.conditional_branch_edges",
    "tl.Trace.conditional_edge_call_indices",
    "tl.Trace.conditional_elif_entry_edges",
    "tl.Trace.conditional_else_entry_edges",
    "tl.Trace.conditional_records",
    "tl.Trace.conditional_then_entry_edges",
    "tl.Trace._current_func_barcode",
    "tl.Trace.detach_hooks",
    "tl.Trace.detach_saved_activations",
    "tl.Trace.recurrence_detection",
    "tl.Trace.do",
    "tl.Trace.emit_nvtx",
    "tl.Trace.equivalent_ops",
    "tl.Trace.find_layers",
    "tl.Trace.find_sites",
    "tl.Trace.first_nonfinite",
    "tl.Trace.flops_by_type",
    "tl.Trace.fork",
    "tl.Trace.forward_source_line",
]

audit_touch_items("Layer and module access", ITEMS, AUDIT_CONTEXT)

## Graph and recurrence metadata

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.Trace.grad_fn_logs",
    "tl.Trace.grad_fn_order",
    "tl.Trace.grad_fns",
    "tl.Trace.grad_transform",
    "tl.Trace.grad_transform_repr",
    "tl.Trace.grads_to_save",
    "tl.Trace.graph_shape_hash",
    "tl.Trace.has_backward_pass",
    "tl.Trace.has_conditional_branching",
    "tl.Trace.has_grads",
    "tl.Trace.input_id",
    "tl.Trace.input_layers",
    "tl.Trace.input_annotations",
    "tl.Trace.input_shape_hash",
    "tl.Trace.internal_source_ops",
    "tl.Trace.internally_terminated_bool_ops",
    "tl.Trace.internal_sink_ops",
    "tl.Trace.intervention_ready",
    "tl.Trace.intervention_spec",
    "tl.Trace.io_format_version",
    "tl.Trace.is_appended",
    "tl.Trace.is_branching",
    "tl.Trace.is_recurrent",
    "tl.Trace.keep_unsaved_layers",
    "tl.Trace.last_run_ctx",
    "tl.Trace.last_run_records",
    "tl.Trace.layer_dict_all_keys",
    "tl.Trace.layer_dict_main_keys",
    "tl.Trace.layer_labels",
    "tl.Trace.layer_labels",
    "tl.Trace.op_labels",
    "tl.Trace.layer_list",
    "tl.Trace.layer_logs",
    "tl.Trace.layer_num_calls",
    "tl.Trace.layers",
    "tl.Trace.layers_with_params",
]

audit_touch_items("Graph and recurrence metadata", ITEMS, AUDIT_CONTEXT)

## Intervention and replay

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.Trace.ops_with_saved_outs",
    "tl.Trace.ops_with_saved_grads",
    "tl.Trace.load",
    "tl.Trace.log_backward",
    "tl.Trace.capture_mode",
    "tl.Trace.macs_by_type",
    "tl.Trace.manual_tensor_connections",
    "tl.Trace.mark_layer_depths",
    "tl.Trace.max_recurrent_loops",
    "tl.Trace.model_name",
    "tl.Trace.module_filter",
    "tl.Trace.modules",
    "tl.Trace.name",
    "tl.Trace.num_context_lines",
    "tl.Trace.num_grad_fns",
    "tl.Trace.num_grad_fns_without_op",
    "tl.Trace.num_ops",
    "tl.Trace.num_streamed_ops",
    "tl.Trace.num_saved_ops",
    "tl.Trace.num_tensors",
    "tl.Trace.observer_spans",
    "tl.Trace.ledger",
    "tl.Trace.orphan_ops",
    "tl.Trace.output_device",
    "tl.Trace.output_layers",
    "tl.Trace.param_logs",
    "tl.Trace.params",
    "tl.Trace.parent_run",
    "tl.Trace.end_time",
    "tl.Trace.start_time",
    "tl.Trace.preview_fastlog",
    "tl.Trace.raise_on_nan",
    "tl.Trace.random_seed",
    "tl.Trace.recording_backward",
    "tl.Trace.recording_kept",
    "tl.Trace.relationship_evidence",
]

audit_touch_items("Intervention and replay", ITEMS, AUDIT_CONTEXT)

## Export and validation

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.Trace.release_param_refs",
    "tl.Trace.remove",
    "tl.Trace.render_dagua_graph",
    "tl.Trace.draw",
    "tl.Trace.replace_run_state_from",
    "tl.Trace.replay",
    "tl.Trace.replay_from",
    "tl.Trace.report_values",
    "tl.Trace.rerun",
    "tl.Trace.resolve_sites",
    "tl.Trace.root_module",
    "tl.Trace.run_state",
    "tl.Trace.save",
    "tl.Trace.save_arg_values",
    "tl.Trace.save_grads",
    "tl.Trace.save_intervention",
    "tl.Trace.save_new_outs",
    "tl.Trace.save_raw_outs",
    "tl.Trace.save_raw_grads",
    "tl.Trace.save_rng_states",
    "tl.Trace.save_code_context",
    "tl.Trace.saved_out_memory",
    "tl.Trace.saved_out_memory_str",
    "tl.Trace.set",
    "tl.Trace.show",
    "tl.Trace.draw_backward",
    "tl.Trace.model_class",
    "tl.Trace.model_id",
    "tl.Trace.streaming_pass_logs",
    "tl.Trace.suggest",
    "tl.Trace.summary",
    "tl.Trace.cleanup_duration",
    "tl.Trace.forward_duration",
    "tl.Trace.func_calls_duration",
    "tl.Trace.overhead_duration",
    "tl.Trace.setup_duration",
]

audit_touch_items("Export and validation", ITEMS, AUDIT_CONTEXT)

## Private state fields

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.Trace.duration",
    "tl.Trace.to_csv",
    "tl.Trace.to_dagua_graph",
    "tl.Trace.to_json",
    "tl.Trace.to_pandas",
    "tl.Trace.to_parquet",
    "tl.Trace.total_out_memory",
    "tl.Trace.total_out_memory_str",
    "tl.Trace.autograd_saved_memory",
    "tl.Trace.total_flops",
    "tl.Trace.total_flops_backward",
    "tl.Trace.total_flops_forward",
    "tl.Trace.total_macs",
    "tl.Trace.total_macs_backward",
    "tl.Trace.total_macs_forward",
    "tl.Trace.num_layers_with_params",
    "tl.Trace.num_param_tensors",
    "tl.Trace.num_params",
    "tl.Trace.num_params_frozen",
    "tl.Trace.param_memory",
    "tl.Trace.param_memory_str",
    "tl.Trace.num_params_trainable",
    "tl.Trace.train_mode",
    "tl.Trace.uncalled_modules",
    "tl.Trace.unlogged_ops",
    "tl.Trace.unsupported_ops",
    "tl.Trace.validate_forward_pass",
    "tl.Trace.validate_saved_outs",
    "tl.Trace.verbose",
    "tl.Trace.visualization_field_audit",
    "tl.Trace.param_hash_quick",
    "tl.Trace.param_hash_full",
]

audit_touch_items("Private state fields", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.Trace",
    "tl.Trace.DEFAULT_FILL_STATE",
    "tl.Trace.FIELD_FORK_POLICY",
    "tl.Trace.PORTABLE_STATE_SPEC",
    "tl.Trace._out_hash_cache",
    "tl.Trace._out_recipe_revision",
    "tl.Trace._layers_logged",
    "tl.Trace._layers_saved",
    "tl.Trace._append_sequence_id",
    "tl.Trace._final_to_raw_layer_labels",
    "tl.Trace._grad_layer_nums_to_save",
    "tl.Trace._has_direct_writes",
    "tl.Trace._intervention_spec",
    "tl.Trace._last_hook_handle_ids",
    "tl.Trace._layer_counter",
    "tl.Trace._layer_num_to_lookup_keys_dict",
    "tl.Trace._layer_nums_to_save",
    "tl.Trace._layers_where_internal_branches_merge_with_input",
    "tl.Trace._lookup_keys_to_layer_num_dict",
    "tl.Trace._tracing_finished",
    "tl.Trace._raw_layer_dict",
    "tl.Trace._raw_layer_labels_list",
    "tl.Trace._raw_layer_type_counter",
    "tl.Trace._raw_to_final_layer_labels",
    "tl.Trace._source_code_blob",
    "tl.Trace._source_model_ref",
    "tl.Trace._spec_revision",
    "tl.Trace._unsaved_layers_lookup_keys",
    "tl.Trace._warned_direct_write",
    "tl.Trace._warned_mutate_in_place",
    "tl.Trace.out_postfunc",
    "tl.Trace._out_transform_repr",
    "tl.Trace.out_transform",
    "tl.Trace.add_node_overlay",
    "tl.Trace.animate_ops",
    "tl.Trace.append_run_state_from",
    "tl.Trace.attach_hooks",
    "tl.Trace.backward_memory_backend",
    "tl.Trace.backward_num_calls",
    "tl.Trace.backward_peak_memory",
    "tl.Trace.backward_root_grad_fn_id",
    "tl.Trace.buffer_layers",
    "tl.Trace.buffer_num_calls",
    "tl.Trace.buffers",
    "tl.Trace.capture_cache_hit",
    "tl.Trace.capture_cache_key",
    "tl.Trace.capture_cache_path",
    "tl.Trace.capture_args_template",
    "tl.Trace.trace_annotations",
    "tl.Trace.check_metadata_invariants",
    "tl.Trace.cleanup",
    "tl.Trace.clear_hooks",
    "tl.Trace.conditional_arm_entry_edges",
    "tl.Trace.conditional_branch_edges",
    "tl.Trace.conditional_edge_call_indices",
    "tl.Trace.conditional_elif_entry_edges",
    "tl.Trace.conditional_else_entry_edges",
    "tl.Trace.conditional_records",
    "tl.Trace.conditional_then_entry_edges",
    "tl.Trace._current_func_barcode",
    "tl.Trace.detach_hooks",
    "tl.Trace.detach_saved_activations",
    "tl.Trace.recurrence_detection",
    "tl.Trace.do",
    "tl.Trace.emit_nvtx",
    "tl.Trace.equivalent_ops",
    "tl.Trace.find_layers",
    "tl.Trace.find_sites",
    "tl.Trace.first_nonfinite",
    "tl.Trace.flops_by_type",
    "tl.Trace.fork",
    "tl.Trace.forward_source_line",
    "tl.Trace.grad_fn_logs",
    "tl.Trace.grad_fn_order",
    "tl.Trace.grad_fns",
    "tl.Trace.grad_transform",
    "tl.Trace.grad_transform_repr",
    "tl.Trace.grads_to_save",
    "tl.Trace.graph_shape_hash",
    "tl.Trace.has_backward_pass",
    "tl.Trace.has_conditional_branching",
    "tl.Trace.has_grads",
    "tl.Trace.input_id",
    "tl.Trace.input_layers",
    "tl.Trace.input_annotations",
    "tl.Trace.input_shape_hash",
    "tl.Trace.internal_source_ops",
    "tl.Trace.internally_terminated_bool_ops",
    "tl.Trace.internal_sink_ops",
    "tl.Trace.intervention_ready",
    "tl.Trace.intervention_spec",
    "tl.Trace.io_format_version",
    "tl.Trace.is_appended",
    "tl.Trace.is_branching",
    "tl.Trace.is_recurrent",
    "tl.Trace.keep_unsaved_layers",
    "tl.Trace.last_run_ctx",
    "tl.Trace.last_run_records",
    "tl.Trace.layer_dict_all_keys",
    "tl.Trace.layer_dict_main_keys",
    "tl.Trace.layer_labels",
    "tl.Trace.layer_labels",
    "tl.Trace.op_labels",
    "tl.Trace.layer_list",
    "tl.Trace.layer_logs",
    "tl.Trace.layer_num_calls",
    "tl.Trace.layers",
    "tl.Trace.layers_with_params",
    "tl.Trace.ops_with_saved_outs",
    "tl.Trace.ops_with_saved_grads",
    "tl.Trace.load",
    "tl.Trace.log_backward",
    "tl.Trace.capture_mode",
    "tl.Trace.macs_by_type",
    "tl.Trace.manual_tensor_connections",
    "tl.Trace.mark_layer_depths",
    "tl.Trace.max_recurrent_loops",
    "tl.Trace.model_name",
    "tl.Trace.module_filter",
    "tl.Trace.modules",
    "tl.Trace.name",
    "tl.Trace.num_context_lines",
    "tl.Trace.num_grad_fns",
    "tl.Trace.num_grad_fns_without_op",
    "tl.Trace.num_ops",
    "tl.Trace.num_streamed_ops",
    "tl.Trace.num_saved_ops",
    "tl.Trace.num_tensors",
    "tl.Trace.observer_spans",
    "tl.Trace.ledger",
    "tl.Trace.orphan_ops",
    "tl.Trace.output_device",
    "tl.Trace.output_layers",
    "tl.Trace.param_logs",
    "tl.Trace.params",
    "tl.Trace.parent_run",
    "tl.Trace.end_time",
    "tl.Trace.start_time",
    "tl.Trace.preview_fastlog",
    "tl.Trace.raise_on_nan",
    "tl.Trace.random_seed",
    "tl.Trace.recording_backward",
    "tl.Trace.recording_kept",
    "tl.Trace.relationship_evidence",
    "tl.Trace.release_param_refs",
    "tl.Trace.remove",
    "tl.Trace.render_dagua_graph",
    "tl.Trace.draw",
    "tl.Trace.replace_run_state_from",
    "tl.Trace.replay",
    "tl.Trace.replay_from",
    "tl.Trace.report_values",
    "tl.Trace.rerun",
    "tl.Trace.resolve_sites",
    "tl.Trace.root_module",
    "tl.Trace.run_state",
    "tl.Trace.save",
    "tl.Trace.save_arg_values",
    "tl.Trace.save_grads",
    "tl.Trace.save_intervention",
    "tl.Trace.save_new_outs",
    "tl.Trace.save_raw_outs",
    "tl.Trace.save_raw_grads",
    "tl.Trace.save_rng_states",
    "tl.Trace.save_code_context",
    "tl.Trace.saved_out_memory",
    "tl.Trace.saved_out_memory_str",
    "tl.Trace.set",
    "tl.Trace.show",
    "tl.Trace.draw_backward",
    "tl.Trace.model_class",
    "tl.Trace.model_id",
    "tl.Trace.streaming_pass_logs",
    "tl.Trace.suggest",
    "tl.Trace.summary",
    "tl.Trace.cleanup_duration",
    "tl.Trace.forward_duration",
    "tl.Trace.func_calls_duration",
    "tl.Trace.overhead_duration",
    "tl.Trace.setup_duration",
    "tl.Trace.duration",
    "tl.Trace.to_csv",
    "tl.Trace.to_dagua_graph",
    "tl.Trace.to_json",
    "tl.Trace.to_pandas",
    "tl.Trace.to_parquet",
    "tl.Trace.total_out_memory",
    "tl.Trace.total_out_memory_str",
    "tl.Trace.autograd_saved_memory",
    "tl.Trace.total_flops",
    "tl.Trace.total_flops_backward",
    "tl.Trace.total_flops_forward",
    "tl.Trace.total_macs",
    "tl.Trace.total_macs_backward",
    "tl.Trace.total_macs_forward",
    "tl.Trace.num_layers_with_params",
    "tl.Trace.num_param_tensors",
    "tl.Trace.num_params",
    "tl.Trace.num_params_frozen",
    "tl.Trace.param_memory",
    "tl.Trace.param_memory_str",
    "tl.Trace.num_params_trainable",
    "tl.Trace.train_mode",
    "tl.Trace.uncalled_modules",
    "tl.Trace.unlogged_ops",
    "tl.Trace.unsupported_ops",
    "tl.Trace.validate_forward_pass",
    "tl.Trace.validate_saved_outs",
    "tl.Trace.verbose",
    "tl.Trace.visualization_field_audit",
    "tl.Trace.param_hash_quick",
    "tl.Trace.param_hash_full",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")